# MELA-TPU (JAX) on a Colab TPU
Runtime -> Change runtime type -> TPU. Upload `MELA-TPU.zip` when asked. Steps: unzip, check devices, run the CPU-vs-TPU equivalence smoke, time one training step at d=256 B=16 T=512 (2 layers).

In [ ]:
import jax, jax.numpy as jnp, os, sys, time
print(jax.__version__, jax.devices())
from google.colab import files
if not os.path.exists('MELA-TPU'):
    up = files.upload()
    import zipfile; zipfile.ZipFile(list(up.keys())[0]).extractall('.')
sys.path.insert(0, 'MELA-TPU')
from melatpu import core, model

In [ ]:
# 1) small equivalence: TPU vs CPU backend on the same inputs (fp32 matmuls forced)
pass  # precision is scoped inside core._mm (HIGHEST on the transport and chain only); no global flag
cfg = core.config(64, 256, chunk=64, recompute=False)
key = jax.random.PRNGKey(0)
P = core.init_params(key, cfg)
h = jax.random.normal(jax.random.PRNGKey(1), (2, 256, 64), jnp.float32)
n_ev = len(range(cfg['k_event'], 256, cfg['k_event']))
us = [jax.random.uniform(jax.random.PRNGKey(10 + i), (2, cfg['n_walks'], cfg['walk_len'] + 1, 2)) for i in range(n_ev)]
f = jax.jit(lambda P, h, us: core.layer_forward(P, cfg, h, us))
o_tpu, inst = f(P, h, us)
cpu = jax.devices('cpu')[0]
with jax.default_device(cpu):
    o_cpu, _ = jax.jit(lambda P, h, us: core.layer_forward(P, cfg, h, us))(jax.device_put(P, cpu), jax.device_put(h, cpu), jax.device_put(us, cpu))
print('TPU vs CPU rel', float(jnp.abs(jax.device_get(o_tpu) - jax.device_get(o_cpu)).max() / jnp.abs(jax.device_get(o_cpu)).max()))
print({k: float(v) for k, v in inst[0].items()})

In [ ]:
# 2) training-step time at the frozen sizing (d 256, B 16, T 512, 2 layers), single TPU core / device 0
d, T, B, V, layers = 256, 512, 16, 65, 2
cfg = core.config(d, T)
P = model.init_lm(jax.random.PRNGKey(0), V, cfg, layers)
opt, step = model.make_train_step(cfg)
st = opt.init(P)
n_ev = len(range(cfg['k_event'], T, cfg['k_event']))
x = jax.random.randint(jax.random.PRNGKey(2), (B, T), 0, V)
def us_for(i):
    return [[jax.random.uniform(jax.random.PRNGKey(1000 * i + 10 * b + e), (B, cfg['n_walks'], cfg['walk_len'] + 1, 2)) for e in range(n_ev)] for b in range(layers)]
for i in range(2):  # compile + warm
    P, st, loss, insts = step(P, st, x, x, us_for(i)); loss.block_until_ready()
ts = []
for i in range(3):
    t0 = time.perf_counter(); P, st, loss, insts = step(P, st, x, x, us_for(10 + i)); loss.block_until_ready(); ts.append(time.perf_counter() - t0)
print('JAX TPU step s (median of 3):', sorted(ts)[1], '| loss', float(loss))
print({k: float(v) for k, v in insts[0][0].items()})

In [ ]:
# 3) same with the default (bf16-pass) matmul precision -- shows the TPU speed/precision trade
pass  # bf16-pass comparison applies to the non-transport path only; the transport stays HIGHEST
opt, step = model.make_train_step(cfg); st = opt.init(P)
for i in range(2): P, st, loss, insts = step(P, st, x, x, us_for(i)); loss.block_until_ready()
ts = []
for i in range(3):
    t0 = time.perf_counter(); P, st, loss, insts = step(P, st, x, x, us_for(20 + i)); loss.block_until_ready(); ts.append(time.perf_counter() - t0)
print('JAX TPU step s, default precision:', sorted(ts)[1], '| orth_drift', float(insts[0][0]['orth_drift']), '| hol_norm', float(insts[0][0]['hol_norm']))